# GigaAM: микрофон → текст локально (Jupyter / VS Code)

Запись с микрофона компьютера, где работает Python-kernel. Start начинает запись, Stop завершает её и запускает распознавание целого фрагмента. Автостоп: не позднее 30 секунд; WAV не больше 5 MB. Частичных результатов во время записи нет. Одна мультиязычная модель, без принудительного языка и без меток языка в результате.

ASR работает локально: аудио не отправляется в STT API. Интернет нужен при установке зависимостей и однократной загрузке весов с проверкой SHA256. Это **не** Colab: удалённый Jupyter/SSH/WSL-кernel обращается к микрофону сервера/своего окружения, а не браузера. Для ноутбука используйте локальный kernel на той же ОС, что и микрофон.

Временные WAV удаляются после распознавания и при обработанных ошибках; звук не встраивается в notebook. Принудительное завершение ОС/питания может оставить временный каталог. Текст виден в виджете и может попасть в сохранённое состояние notebook: перед публикацией очистите outputs и widget state. Экспорт TXT выполняется только по вашему клику. Записывайте только с согласия участников.

## 1. Подготовка Python 3.12 и программ

Для kernel создайте отдельное окружение `.venv-notebook`. ASR будет установлен во **второе** окружение `.venv-gigaam-mic`, чтобы Torch/NumPy не меняли зависимости Jupyter.

Windows (PowerShell): установите Python 3.12 с [python.org](https://www.python.org/downloads/windows/) или `winget install Python.Python.3.12`; установите [Git](https://git-scm.com/downloads/win) и FFmpeg (`winget install Gyan.FFmpeg`). Перезапустите терминал. PortAudio обычно входит в pip-пакет sounddevice. В Settings → Privacy & security → Microphone разрешите доступ настольным приложениям.

```powershell
py -3.12 -m venv .venv-notebook
.venv-notebook\Scripts\python -m pip install jupyterlab ipykernel ipywidgets sounddevice
.venv-notebook\Scripts\python -m ipykernel install --user --name gigaam-mic --display-name "GigaAM mic Python 3.12"
.venv-notebook\Scripts\python -m jupyterlab
```

Ubuntu 24.04 / Debian с доступным Python 3.12 (не заменяйте системный Python):

```bash
sudo apt update
sudo apt install python3.12 python3.12-venv git ffmpeg libportaudio2
python3.12 -m venv .venv-notebook
.venv-notebook/bin/python -m pip install jupyterlab ipykernel ipywidgets sounddevice
.venv-notebook/bin/python -m ipykernel install --user --name gigaam-mic --display-name "GigaAM mic Python 3.12"
.venv-notebook/bin/python -m jupyterlab
```

Если дистрибутив не содержит Python 3.12, установите его параллельно через [официальный Python](https://www.python.org/downloads/source/) или [uv python install 3.12](https://docs.astral.sh/uv/guides/install-python/) и создайте kernel на нём. На Linux проверьте выбранный input в PipeWire/PulseAudio; не запускайте Jupyter от root.

macOS (установленный Homebrew): `brew install python@3.12 git ffmpeg portaudio`, затем используйте команды Linux для venv/kernel, заменив пакетную установку. Разрешите микрофон Terminal/VS Code в System Settings → Privacy & Security → Microphone. CPU поддерживается по возможности наличия колёс; MPS здесь не используется.

В VS Code установите расширения Python и Jupyter от Microsoft, откройте этот `.ipynb`, затем Select Kernel → `GigaAM mic Python 3.12`. Команда запуска JupyterLab в этом случае не нужна. В JupyterLab выберите тот же kernel. Если кнопки не отображаются, проверьте ipywidgets в kernel и поддержку widgets в интерфейсе; перезапустите kernel после установки.

Проверенный dependency snapshot проекта относится к **Python 3.12 / Linux x86_64**. Windows/macOS инструкции даны для локального запуска, но полная совместимость этого snapshot там не подтверждена. Ошибку «No matching distribution» не обходите бесконтрольным обновлением всего окружения. Нужны несколько GB диска под окружение/веса и RAM для модели; при нехватке памяти закройте приложения или используйте машину с большим RAM.

Источники: [sounddevice / PortAudio](https://python-sounddevice.readthedocs.io/en/latest/installation.html), [PyTorch 2.10 wheels](https://pytorch.org/get-started/previous-versions/).


In [ ]:
# Проверка kernel и программ (не устанавливает ничего).
import sys, shutil, platform
from pathlib import Path
assert sys.version_info[:2] == (3, 12), "Выберите kernel Python 3.12"
for tool in ("git", "ffmpeg"):
    assert shutil.which(tool), f"{tool} не найден в PATH; установите и перезапустите kernel"
import sounddevice as sd
import ipywidgets as widgets
from IPython.display import display, HTML
print("Kernel:", sys.executable, "| ОС:", platform.system())
print("PortAudio:", sd.get_portaudio_version()[1])


## 2. Отдельное окружение ASR и фиксированная версия сервиса

Следующая ячейка загружает исполняемый код из указанного публичного репозитория и зависимости. Прочитайте источники перед запуском. Используется commit `dd7b94a6c680d1ca891eeaa799bd68d70cf2559e`; текущий checkout проекта не изменяется. Каталог создаётся в домашней папке. Повторный запуск использует тот же checkout и проверяет commit/отсутствие изменений отслеживаемых файлов.

По умолчанию `DEVICE = "cpu"`. Для Windows/Linux x86_64 с NVIDIA и подходящим драйвером можно установить `DEVICE = "cuda"`: будут выбраны официальные cu128 wheels. Если CUDA недоступна после установки, inference автоматически использует CPU. Смена DEVICE требует повторного запуска установочной ячейки. Загрузка и запуск модели могут занять заметное время; скорость и точность не гарантируются.


In [ ]:
import subprocess, os, venv
SERVICE_URL = "https://github.com/AssylbekGizatov/teams-local-transcriber.git"
SERVICE_REVISION = "dd7b94a6c680d1ca891eeaa799bd68d70cf2559e"
WORK = Path.home() / ".gigaam-microphone-notebook"
REPO = WORK / ("service-" + SERVICE_REVISION[:12])
ASR_ENV = WORK / ".venv-gigaam-mic"
ASR_PY = ASR_ENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
MODEL = WORK / "models" / "gigaam"
DEVICE = "cpu"  # "cuda" для Windows/Linux x86_64; фактическая доступность проверяется отдельно
assert DEVICE in {"cpu", "cuda"}
if DEVICE == "cuda":
    assert platform.system() in {"Linux", "Windows"} and platform.machine() in {"x86_64", "AMD64"}
WORK.mkdir(parents=True, exist_ok=True)
def run_setup(args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True, timeout=1800)
if not REPO.exists():
    run_setup(["git", "clone", "--no-checkout", SERVICE_URL, REPO])
    run_setup(["git", "checkout", "--detach", SERVICE_REVISION], cwd=REPO)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == SERVICE_REVISION, "Неверная ревизия; используйте новый каталог WORK"
assert not subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True).strip(), "Checkout изменён; используйте новый каталог WORK"
if not ASR_PY.exists():
    venv.EnvBuilder(with_pip=True).create(ASR_ENV)
assert subprocess.check_output([str(ASR_PY), "-c", "import sys;print(sys.version_info[:2])"], text=True).strip() == "(3, 12)"
run_setup([ASR_PY, "-m", "pip", "install", "--upgrade", "pip"])
wheels = "https://download.pytorch.org/whl/" + ("cu128" if DEVICE == "cuda" else "cpu")
wheel_version = "2.10.0" if platform.system() == "Darwin" else "2.10.0+" + ("cu128" if DEVICE == "cuda" else "cpu")
torch_args = [ASR_PY, "-m", "pip", "install", f"torch=={wheel_version}", f"torchaudio=={wheel_version}"]
# На macOS официальные CPU wheels поставляются через PyPI.
if platform.system() != "Darwin":
    torch_args += ["--index-url", wheels]
run_setup(torch_args)
run_setup([ASR_PY, "-m", "pip", "install", "-r", REPO / "requirements-asr.lock"])
run_setup([ASR_PY, "-m", "pip", "check"])
print("Отдельное ASR-окружение готово:", ASR_PY)


## 3. Однократная загрузка модели и локальная проверка

Скачиваются только фиксированные файлы модели `ai-sage/GigaAM-Multilingual`, лицензия и provenance; скрипт проверяет SHA256. Повторный запуск проверяет уже скачанные файлы. После этой ячейки интернет для распознавания не нужен. Модель загружается в новый локальный процесс для каждого фрагмента, что экономит память между записями, но добавляет задержку.


In [ ]:
run_setup([ASR_PY, REPO / "scripts/download_models.py", "--backend", "gigaam", "--output", MODEL], cwd=REPO)
probe = subprocess.check_output([str(ASR_PY), "-c", "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"], text=True).strip()
INFERENCE_DEVICE = "cuda" if DEVICE == "cuda" and probe == "cuda" else "cpu"
print("Inference device:", INFERENCE_DEVICE)
if DEVICE == "cuda" and INFERENCE_DEVICE == "cpu":
    print("CUDA недоступна; используется CPU.")


## 4. Выберите микрофон

Запустите ячейку, найдите ID нужного устройства с input channels. Оставьте `INPUT_DEVICE = None` для системного устройства по умолчанию либо укажите числовой ID. Частота берётся из default samplerate устройства; локальный backend затем приводит WAV к mono 16 kHz. Если устройство занято, закройте приложения, использующие микрофон, или выберите другое.


In [ ]:
print(sd.query_devices())
INPUT_DEVICE = None  # например: 1


## 5. Start / Stop → итоговый текст

Нажмите Start, произнесите фразу, затем Stop. Stop останавливает микрофон и запускает ASR; на лимите это происходит автоматически. Новая запись доступна только после завершения ASR. «Закрыть / отменить» останавливает микрофон и прерывает текущий процесс распознавания. Если появилась ошибка, проверьте разрешения, устройство, FFmpeg, модель и память; перезапустите эту ячейку после исправления.

Временные файлы удаляются в `finally`/контекстном менеджере. Не завершайте процесс kernel принудительно во время работы: сначала нажмите «Закрыть / отменить». Текст не экспортируется автоматически.


In [ ]:
# MIC_CONTROLLER: self-contained; no notebook helper download needed.
import atexit, base64, json, signal, tempfile, threading, time, wave

MAX_SECONDS = 30
MAX_BYTES = 5_000_000
INFERENCE_TIMEOUT = 600
WORKER_CODE = r"""
import json, sys
from pathlib import Path
from app.asr import ASRSettings, LocalASR
settings = ASRSettings(backend="gigaam", model_path=Path(sys.argv[2]),
    device=sys.argv[3], max_audio_seconds=30, max_input_bytes=5_000_000,
    ffmpeg_timeout_seconds=60, temp_root=Path(sys.argv[1]).parent)
segments = LocalASR(settings).transcribe(Path(sys.argv[1]))
print(json.dumps({"text": " ".join(s["text"] for s in segments)}, ensure_ascii=False))
"""

class MicrophoneNotebook:
    def __init__(self):
        self.stop_event = threading.Event()
        self.cancel_event = threading.Event()
        self.thread = None
        self.closed = False
        self.start_button = widgets.Button(description="Start", button_style="success")
        self.stop_button = widgets.Button(description="Stop", button_style="warning", disabled=True)
        self.close_button = widgets.Button(description="Закрыть / отменить")
        self.download_button = widgets.Button(description="Подготовить TXT", disabled=True)
        self.status = widgets.Label(value="Готово к записи")
        self.result = widgets.Textarea(value="", description="Текст:", layout=widgets.Layout(width="100%", height="180px"))
        self.download = widgets.HTML(value="")
        self.start_button.on_click(lambda _: self.start())
        self.stop_button.on_click(lambda _: self.stop_event.set())
        self.close_button.on_click(lambda _: self.close())
        self.download_button.on_click(lambda _: self.prepare_download())
        self.ui = widgets.VBox([widgets.HBox([self.start_button, self.stop_button, self.close_button]), self.status, self.result, self.download_button, self.download])
        atexit.register(self.close)

    def start(self):
        if self.closed or (self.thread and self.thread.is_alive()):
            return
        self.stop_event.clear()
        self.cancel_event.clear()
        self.result.value = ""
        self.download.value = ""
        self.download_button.disabled = True
        self.start_button.disabled = True
        self.stop_button.disabled = False
        self.thread = threading.Thread(target=self._record_and_transcribe, daemon=True)
        self.thread.start()

    def _record_and_transcribe(self):
        pcm = bytearray()
        callback_error = []
        try:
            device = sd.query_devices(INPUT_DEVICE, "input")
            rate = int(round(float(device["default_samplerate"])))
            if not 8000 <= rate <= 384000:
                raise ValueError("Неожиданная частота микрофона")
            sd.check_input_settings(device=INPUT_DEVICE, channels=1, dtype="int16", samplerate=rate)
            # Leave room for WAV header; keep full int16 samples only.
            cap = min(rate * MAX_SECONDS * 2, MAX_BYTES - 4096)
            cap -= cap % 2
            deadline = time.monotonic() + MAX_SECONDS
            def callback(indata, frames, timing, status):
                if self.stop_event.is_set() or self.cancel_event.is_set():
                    raise sd.CallbackStop
                try:
                    if status:
                        callback_error.append("Сбой/переполнение аудиовхода; повторите запись")
                        self.stop_event.set()
                        raise sd.CallbackStop
                    remaining = cap - len(pcm)
                    pcm.extend(bytes(indata)[:remaining])
                    if len(pcm) >= cap or time.monotonic() >= deadline:
                        self.stop_event.set()
                        raise sd.CallbackStop
                except sd.CallbackStop:
                    raise
                except Exception:
                    callback_error.append("Ошибка аудиозаписи")
                    self.stop_event.set()
                    raise sd.CallbackAbort
            self.status.value = "Запись… нажмите Stop (максимум 30 секунд / 5 MB)"
            with sd.RawInputStream(device=INPUT_DEVICE, samplerate=rate, channels=1,
                    dtype="int16", blocksize=1024, callback=callback,
                    finished_callback=self.stop_event.set):
                self.stop_event.wait(MAX_SECONDS)
                self.stop_event.set()
            self.stop_button.disabled = True
            if self.cancel_event.is_set():
                return
            if callback_error:
                raise RuntimeError(callback_error[0])
            if len(pcm) < rate // 5 * 2:
                raise ValueError("Запись слишком короткая: произнесите фразу дольше 0.2 секунды")
            self.status.value = "Микрофон выключен. Локальное распознавание…"
            with tempfile.TemporaryDirectory(prefix="gigaam-microphone-") as temp:
                wav_path = Path(temp) / "microphone.wav"
                with wave.open(str(wav_path), "wb") as audio:
                    audio.setnchannels(1)
                    audio.setsampwidth(2)
                    audio.setframerate(rate)
                    audio.writeframes(pcm)
                pcm.clear()
                if wav_path.stat().st_size > MAX_BYTES:
                    raise ValueError("Превышен лимит WAV")
                env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUTF8="1")
                command = [str(ASR_PY), "-c", WORKER_CODE, str(wav_path), str(MODEL), INFERENCE_DEVICE]
                with subprocess.Popen(command, cwd=REPO, env=env, stdin=subprocess.DEVNULL,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding="utf-8",
                        start_new_session=(os.name != "nt"),
                        creationflags=(subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0)) as process:
                    started = time.monotonic()
                    try:
                        while True:
                            if self.cancel_event.is_set():
                                raise InterruptedError("Отменено")
                            if time.monotonic() - started > INFERENCE_TIMEOUT:
                                raise TimeoutError("Лимит распознавания 600 секунд; попробуйте короче или более мощную машину")
                            try:
                                stdout, stderr = process.communicate(timeout=0.2)
                                break
                            except subprocess.TimeoutExpired:
                                continue
                        if process.returncode:
                            # No audio/text/traceback is printed automatically.
                            raise RuntimeError("ASR завершился с ошибкой: проверьте зависимости, модель, FFmpeg и RAM")
                        payload = json.loads(stdout.strip().splitlines()[-1])
                        self.result.value = payload["text"]
                        self.download_button.disabled = not bool(self.result.value)
                        self.status.value = "Готово" if self.result.value else "Текст не получен; попробуйте другую фразу"
                    finally:
                        if process.poll() is None:
                            self._kill_process_tree(process)
                        process.communicate()
        except InterruptedError:
            self.status.value = "Отменено"
        except Exception as error:
            self.status.value = f"Ошибка: {error}"
        finally:
            pcm.clear()
            self.stop_event.set()
            self.stop_button.disabled = True
            self.start_button.disabled = self.closed
            if self.closed:
                self.status.value = "Закрыто. Для новой записи выполните эту ячейку заново."

    @staticmethod
    def _kill_process_tree(process):
        if os.name == "nt":
            # FFmpeg is a child too; terminate the tree before deleting its WAVs.
            try:
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=10, check=False)
            finally:
                if process.poll() is None:
                    process.kill()
        else:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    def prepare_download(self):
        if not self.result.value:
            return
        encoded = base64.b64encode(self.result.value.encode("utf-8")).decode("ascii")
        self.download.value = '<a download="gigaam-transcript.txt" href="data:text/plain;charset=utf-8;base64,' + encoded + '">Скачать TXT</a>'
        # Some VS Code renderers block data links; copying text remains available.

    def close(self):
        self.closed = True
        self.cancel_event.set()
        self.stop_event.set()
        if self.thread and self.thread is not threading.current_thread():
            self.thread.join(timeout=5)
        self.start_button.disabled = True
        self.stop_button.disabled = True
        self.status.value = "Закрыто" if not self.thread or not self.thread.is_alive() else "Остановка…"
        atexit.unregister(self.close)

if "microphone" in globals():
    microphone.close()
    if microphone.thread and microphone.thread.is_alive():
        raise RuntimeError("Предыдущая запись ещё закрывается; подождите и повторите ячейку")
microphone = MicrophoneNotebook()
display(microphone.ui)


## 6. Фразы для самостоятельной проверки

Произносите отдельно, затем смешанную фразу. Это примеры входа, не эталон качества и не результаты выполненного теста.

- RU: «Сегодня мы обсудим план проекта и назначим следующую встречу на пятницу».
- KK: «Бүгін біз жобаның жоспарын талқылаймыз және келесі кездесудің уақытын белгілейміз».
- EN: “Today we will review the project plan and schedule the next meeting for Friday.”
- Mixed: «Бүгін meeting agenda бойынша сөйлесеміз, затем проверим сроки, and send the final report».

Виджет показывает только итоговый текст. Нажмите «Подготовить TXT», затем «Скачать TXT», если файл действительно нужен. Некоторые версии VS Code блокируют data-ссылки: тогда скопируйте текст вручную в новый `.txt` файл, либо откройте notebook в локальном JupyterLab. Аудио при этом не сохраняется.

### Если что-то не работает

- Нет микрофона / PortAudio error: проверьте устройство, разрешения ОС, PortAudio на Linux и что kernel локальный. Browser microphone permission для sounddevice не используется.
- Несовпадение частоты: выберите другой input ID; FFmpeg нормализует аудио после записи, но устройство должно поддерживать выбранную native частоту и mono input.
- Ошибка установки: проверьте Python 3.12 и ОС/архитектуру. Не ставьте ASR зависимости в kernel Jupyter. Исходный snapshot проверен на Linux x86_64.
- Ошибка/долгое ASR: сначала дождитесь первой загрузки; проверьте FFmpeg и завершение загрузки модели. При превышении 600 секунд процесс прекращается, временное аудио удаляется. Попробуйте короткий фрагмент.
- После записи экран не меняется: проверьте соединение Jupyter с kernel. Нажмите «Закрыть / отменить»; повторно выполните ячейку виджета. При принудительном завершении kernel возможны остатки `gigaam-microphone-*` в системной temp-папке.

Окружения/веса сохраняются в `~/.gigaam-microphone-notebook` для повторного использования. Для удаления сначала закройте notebook/kernel и удалите эту папку самостоятельно. Не публикуйте notebook с личным распознанным текстом.
